In [ ]:
import numpy as np
import os
import scipy.integrate as integrate
import matplotlib.pyplot as plt

In [ ]:
delta = 5e-2

a_t = 0
b_t = 0.5
a_xi = - delta
b_xi = delta


number_of_grid_xi = 5000
number_of_grid_t = 5000

N  = (b_xi - a_xi)*number_of_grid_xi
Nt = (b_t - a_t)*number_of_grid_t
h_ = (b_xi - a_xi)/N
k = (b_t - a_t)/Nt

t = np.arange(a_t, b_t, k)
xi = np.arange(a_xi, b_xi, h_)


In [ ]:
xi_interface = np.array([0])
if (xi.shape[0] % 2) == 0:
    idx_xi_interface = np.floor((xi_interface+delta)*len(xi)*10).astype(int) - 1
else:
    idx_xi_interface = np.floor((xi_interface+delta)*len(xi)*10).astype(int)
print(idx_xi_interface)

In [ ]:
xi[idx_xi_interface]

In [ ]:
C_star = 0.1
eta = 1

d1 = 1
d2 = 1e-2

k1 = 1
k2 = 1

n = np.arange(1,101).tolist()
inc = 5
dt = k*inc
print("dt: ", dt)

In [ ]:
U_exact = np.zeros((t.shape[0], xi.shape[0]))
U_bar_exact = np.zeros((t.shape[0], xi.shape[0]))
psi_exact = np.zeros((t.shape[0], xi.shape[0]))

dUm_dx_inter = np.zeros((t.shape[0], 1))
dUp_dx_inter = np.zeros((t.shape[0], 1))
dUm_dxx_inter = np.zeros((t.shape[0], 1))
dUp_dxx_inter = np.zeros((t.shape[0], 1))

In [ ]:
cond_ic = xi < 0
xi_m = np.where(cond_ic, xi, np.nan)
xi_m = xi_m[~np.isnan(xi_m)]

xi_p = np.where(cond_ic, np.nan, xi)
xi_p = xi_p[~np.isnan(xi_p)]

In [ ]:
xi_m = np.reshape(xi_m, (xi_m.shape[0], 1))
xi_p = np.reshape(xi_p, (xi_p.shape[0], 1))

t = np.reshape(t, (t.shape[0], 1))

In [ ]:
# Define the interface position and velocity.
h = lambda t, c = C_star, eta = eta : eta * t + c
h_prime = lambda t, eta = eta: np.ones(t.shape) * eta

boundary = lambda t, ic : np.ones(t.shape) * ic

# Evaluate the transformed-coordinate coefficient.
def r(t, d, h=h, hprime=h_prime):
    return hprime(t) + (d/h(t))

# Define the initial concentration profile.
U0 = lambda xi, ic, delta = delta : ic * np.sin(np.pi*(xi+delta)/(2*delta)) + 0.97


Wm_func = lambda xi, Am, Bm, delta = delta : (1 + (xi/delta))*Am - (xi/delta)*Bm
Wp_func = lambda xi, Ap, Bp, delta = delta : (1 - (xi/delta))*Ap + (xi/delta)*Bp

In [ ]:
u_icl = u_icr = 0.02

U_icl = U0(xi_m, u_icl) 
U_icr = U0(xi_p, u_icr) 


U_bcl = boundary(t, U_icl[0])
U_bcr = boundary(t, U_icr[-1])

In [ ]:
U_ic = np.concatenate((U_icl.flatten(), U_icr.flatten()))
U_ic = np.reshape(U_ic, (U_ic.shape[0], 1)).T

In [ ]:

U_bar_exact[0:1,:] = U_ic - 1

U_bar_icl = U_icl - 1
U_bar_icr = U_icr - 1

U_bar_bcl = U_bcl - 1
U_bar_bcr = U_bcr - 1



In [ ]:
# Assemble Fourier coefficients for the transformed solution.
def find_constants(xi_m, xi_p, t, U_icl, U_icr, am, ap, bm, bp, Bm, Bp, h_prime = h_prime, delta = delta, n = n, d1 = d1, d2 = d2):


    I = 0
    J = 0
    K = 0
    L = 0
    M = 0
    N = 0
    O = 0
    
    En_list = []
    Fn_list = []
    Gn_list = []
    Hn_list = []

    for ni in n:

        integrand_En = (np.exp(ap*xi_p)*U_icr - xi_p*Bp/delta)*np.sin(ni*np.pi*xi_p/delta)
        integrand_Fn = (1 - xi_p/delta)*np.sin(ni*np.pi*xi_p/delta)
        integrand_Gn = (np.exp(am*xi_m)*U_icl + xi_m*Bm/delta)*np.sin(ni*np.pi*xi_m/delta)
        integrand_Hn = (1 + xi_m/delta)*np.sin(ni*np.pi*xi_m/delta)

        En = (2/delta)*integrate.cumulative_simpson(integrand_En.flatten(), x = xi_p.flatten())[len(xi_p)-2]
        Fn = (-2/delta)*integrate.cumulative_simpson(integrand_Fn.flatten(), x = xi_p.flatten())[len(xi_p)-2]
        Gn = (2/delta)*integrate.cumulative_simpson(integrand_Gn.flatten(), x = xi_m.flatten())[len(xi_m)-2]
        Hn = (-2/delta)*integrate.cumulative_simpson(integrand_Hn.flatten(), x = xi_m.flatten())[len(xi_m)-2]

        I += Gn*np.exp(-d1*((ni*np.pi/delta)**2)*t)*(ni*np.pi/delta)
        J += Hn*np.exp(-d1*((ni*np.pi/delta)**2)*t)*(ni*np.pi/delta)
        K += En*np.exp(-d2*((ni*np.pi/delta)**2)*t)*(ni*np.pi/delta)
        L += Fn*np.exp(-d2*((ni*np.pi/delta)**2)*t)*(ni*np.pi/delta)

        En_list.append(En)
        Fn_list.append(Fn)
        Gn_list.append(Gn)
        Hn_list.append(Hn)

    M = np.exp(-bp*t)*(h_prime(t) + d2*(-ap) + d2*L - d2/delta)
    N = - np.exp(-bm*t)*(h_prime(t) + d1*(-am) + d1*J + d1/delta)
    O = d2*np.exp(-bp*t)*K - d1*np.exp(-bm*t)*I + d2*np.exp(-bp*t)*Bp/delta + d1*np.exp(-bm*t)*Bm/delta
    

    return np.array(En_list), np.array(Fn_list), np.array(Gn_list), np.array(Hn_list), M, N, O


In [ ]:
def vm_vp_func(xi_m, xi_p, t, Cnm, Cnp, n = n):
    vm = 0
    vp = 0
    dvm_dx = 0
    dvp_dx = 0

    for ni in n:
        vm += Cnm[ni-1]*np.exp(-d1*((ni*np.pi/delta)**2)*t)*np.sin(ni*np.pi*xi_m/delta).T
        vp += Cnp[ni-1]*np.exp(-d2*((ni*np.pi/delta)**2)*t)*np.sin(ni*np.pi*xi_p/delta).T

        dvm_dx += Cnm[ni-1]*np.exp(-d1*((ni*np.pi/delta)**2)*t)*(ni*np.pi/delta)
        dvp_dx += Cnp[ni-1]*np.exp(-d2*((ni*np.pi/delta)**2)*t)*(ni*np.pi/delta)

    return vm, vp, dvm_dx, dvp_dx

In [ ]:
bm_list = []
bp_list = []
aa = 0

In [ ]:
# Advance the solution in time blocks.
for i in range(1,len(t),inc):
    print(i)
    ti = t[i : i + inc ] - aa*dt
    print(ti)

    aa += 1

    print("t i - 1: ", t[i - 1])

    r1 = r(t[i - 1], d1)[0]
    r2 = r(t[i - 1], d2)[0]

    print("r1: ", r1)
    print("r2: ", r2)

    am = r1/(2*d1)
    bm = (r1*r1 + 4*d1*k1) / (4*d1)

    ap = r2/(2*d2)
    bp = (r2*r2 + 4*d2*k2) / (4*d2)

    print("am: ", am)
    print("ap: ", ap) 

    print("bm: ", bm)
    print("bp: ", bp)

    if i - 1 == 0:
        psi_m_ = np.exp(am*xi_m)*U_bar_icl
        psi_p_ = np.exp(ap*xi_p)*U_bar_icr

        psi_exact[0 : 1, :] = np.concatenate((psi_m_, psi_p_)).T

    integrand_Bm = np.exp(-am*delta + bm*ti)*U_bar_bcl[i : i + inc]
    integrand_Bp = np.exp(ap*delta + bp*ti)*U_bar_bcr[i : i + inc]
    Bm = np.mean(integrand_Bm)
    Bp = np.mean(integrand_Bp)


    print("Bm: ", Bm)
    print("Bp: ", Bp)

    Ac = np.mean(np.exp(bm*ti)) / np.mean(np.exp(bp*ti))


    En, Fn, Gn, Hn, M, N, O = find_constants(xi_m, xi_p, ti[0], U_bar_icl, U_bar_icr, am, ap, bm, bp, Bm, Bp)

    Ap = - O / (M + Ac*N)
    Am = Ac * Ap
    print("Ap: ", Ap)
    print("Am: ", Am)

    alpha_m = (Am)/np.mean(np.exp(bm*ti))
    alpha_p = (Ap)/np.mean(np.exp(bp*ti))

    print("alpha_m: ", alpha_m)
    print("alpha_p: ", alpha_p)


    Cnm = Gn + Am*Hn
    Cnp = En + Ap*Fn

    Wm = Wm_func(xi_m, Am, Bm)
    Wp = Wp_func(xi_p, Ap, Bp)

    Wm_mx = np.tile(Wm.flatten(), (len(ti), 1))
    Wp_mx = np.tile(Wp.flatten(), (len(ti), 1))


    Vm, Vp, dVm_dx_interface, dVp_dx_interface = vm_vp_func(xi_m, xi_p, ti, Cnm, Cnp)

    psi_m = Wm_mx + Vm
    psi_p = Wp_mx + Vp

    Ubarm = (np.exp(-bm*ti)*np.exp(-am*xi_m).T)*psi_m
    Ubarp = (np.exp(-bp*ti)*np.exp(-ap*xi_p).T)*psi_p

    psi_soln = np.concatenate((psi_m, psi_p), axis=1)
    U_bar_soln = np.concatenate((Ubarm, Ubarp), axis=1)

    psi_exact[i : i + len(ti), :] = psi_soln
    U_bar_exact[i : i + len(ti), :] = U_bar_soln

    U_bar_icl = Ubarm[len(ti)-1:len(ti),:].T
    U_bar_icr = Ubarp[len(ti)-1:len(ti),:].T

    bm_list.append(bm)
    bp_list.append(bp)

    if i - 1 == 0:
        dVm_dx_interface_t0 = 0
        dVp_dx_interface_t0 = 0

        for ni in n:
            dVm_dx_interface_t0 += Cnm[ni-1]*(ni*np.pi/delta)
            dVp_dx_interface_t0 += Cnp[ni-1]*(ni*np.pi/delta)

        dUm_dx_inter[0, :] = (-am*Am + dVm_dx_interface_t0 + (Am - Bm) / delta)
        dUp_dx_inter[0, :] = (-ap*Ap + dVp_dx_interface_t0 + (Bp - Ap) / delta)
        dUm_dxx_inter[0, :] = (am*am*Am - 2*am*(dVm_dx_interface_t0 + (Am - Bm) / delta))
        dUp_dxx_inter[0, :] = (ap*ap*Ap - 2*ap*(dVp_dx_interface_t0 + (Bp - Ap) / delta))

    dUm_dx_interface = np.exp(-bm*ti)*(-am*Am + dVm_dx_interface + (Am - Bm) / delta)
    dUp_dx_interface = np.exp(-bp*ti)*(-ap*Ap + dVp_dx_interface + (Bp - Ap) / delta)

    dUm_dxx_interface = np.exp(-bm*ti)*(am*am*Am - 2*am*(dVm_dx_interface + (Am - Bm) / delta))
    dUp_dxx_interface = np.exp(-bp*ti)*(ap*ap*Ap - 2*ap*(dVp_dx_interface + (Bp - Ap) / delta))

    dUm_dx_inter[i : i + len(ti), :] = dUm_dx_interface
    dUp_dx_inter[i : i + len(ti), :] = dUp_dx_interface
    dUm_dxx_inter[i : i + len(ti), :] = dUm_dxx_interface
    dUp_dxx_inter[i : i + len(ti), :] = dUp_dxx_interface
    



In [ ]:
U_exact = U_bar_exact + 1

In [ ]:
bm_array = np.array(bm_list)
bp_array = np.array(bp_list)

In [ ]:
plt.plot(range(0,len(bm_array)), bm_array)
plt.ylabel(r"$b_-$")

In [ ]:
plt.plot(bp_array)
plt.ylabel(r"$b_+$")

In [ ]:
ts = [0.0, 0.00025, 0.0005, 0.4]

In [ ]:
fig, axs = plt.subplots(2, 2,figsize=(10,8))


axs[0, 0].plot(xi,U_bar_exact[int((ts[0]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')

axs[0, 0].set_title('$t = $' + str(ts[0]), fontsize = 10)

axs[0, 0].grid()

axs[0, 1].plot(xi,U_bar_exact[int((ts[1]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')

axs[0, 1].set_title('$t = $' + str(ts[1]), fontsize = 10)

axs[0, 1].grid()

axs[1, 0].plot(xi,U_bar_exact[int((ts[2]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')

axs[1, 0].set_title('$t = $'+ str(ts[2]), fontsize = 10)

axs[1, 0].grid()

axs[1, 1].plot(xi,U_bar_exact[int((ts[3]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')

axs[1, 1].set_title('$t = $' + str(ts[3]), fontsize = 10)

axs[1, 1].grid()

for ax in axs.flat:
    ax.set(xlabel=r'$\xi$', ylabel=r'$U(\xi,t)$')
    
plt.legend()
fig.tight_layout()


plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2,figsize=(10,8))


axs[0, 0].plot(xi,U_exact[int((ts[0]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')

axs[0, 0].set_title('$t = $' + str(ts[0]), fontsize = 10)
axs[0, 0].set_ylim([0.95  - 0.002, 1 + 0.002])
axs[0, 0].grid()

axs[0, 1].plot(xi,U_exact[int((ts[1]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')

axs[0, 1].set_title('$t = $' + str(ts[1]), fontsize = 10)
axs[0, 1].set_ylim([0.95  - 0.002, 1 + 0.002])
axs[0, 1].grid()

axs[1, 0].plot(xi,U_exact[int((ts[2]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')

axs[1, 0].set_title('$t = $'+ str(ts[2]), fontsize = 10)
axs[1, 0].set_ylim([0.95  - 0.002, 1 + 0.002])
axs[1, 0].grid()

axs[1, 1].plot(xi,U_exact[int((ts[3]/b_t)*(len(t)-1)),:], linewidth = 2, color = 'blue', label = 'Exact')

axs[1, 1].set_title('$t = $' + str(ts[3]), fontsize = 10)
axs[1, 1].set_ylim([0.95  - 0.002, 1 + 0.002])
axs[1, 1].grid()

for ax in axs.flat:
    ax.set(xlabel=r'$\xi$', ylabel=r'$U(\xi,t)$')
    
plt.legend()
fig.tight_layout()


plt.show()